# NYC Taxi — Spark Transformations, Joins & Partitioning

Dataset: NYC TLC Yellow Taxi trips  
Topics covered:
- Transformations (select, filter, withColumn, UDFs, window functions)
- Join strategies (broadcast, sort-merge, shuffle hash) and when to use each
- Partitioning (repartition vs coalesce, partition pruning, write partitioning)
- Caching and when it pays off
- Reading explain plans

## 0. Setup

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import time

spark = (
    SparkSession.builder
    .appName("NYCTaxi")
    .config("spark.sql.adaptive.enabled", "true")       # AQE on  — Spark 3+
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
BUCKET      = "emr-teaching-mentorship-2026"
SHARED      = f"s3://{BUCKET}/shared/datasets/nyc_taxi"

# Update to your name
STUDENT_NAME = "anastasia"
MY_PREFIX    = f"s3://{BUCKET}/{STUDENT_NAME}"

YELLOW_PATH = f"{SHARED}/yellow_taxi/"   # yellow taxi trips (tpep_* datetime columns)
GREEN_PATH  = f"{SHARED}/green_taxi/"    # green taxi trips  (lpep_* datetime columns)
ZONES_PATH  = f"{SHARED}/taxi_zones.csv" # small lookup: LocationID -> Borough/Zone

## 1. Load the data

In [ ]:
yellow = spark.read.parquet(YELLOW_PATH)
green  = spark.read.parquet(GREEN_PATH)

print("=== Yellow schema ===")
yellow.printSchema()
print("=== Green schema ===")
green.printSchema()

In [ ]:
zones = spark.read.csv(ZONES_PATH, header=True, inferSchema=True)
zones.show(5)
print(f"Yellow: {yellow.count():,}  Green: {green.count():,}  Zones: {zones.count()}")

### 1b — Schema differences: Yellow vs Green

Yellow taxi uses `tpep_pickup_datetime` / `tpep_dropoff_datetime`.  
Green taxi uses `lpep_pickup_datetime` / `lpep_dropoff_datetime`.  
Green also has a `trip_type` column (1=street-hail, 2=dispatch) that yellow doesn't.  

Before unioning them we need to:
1. Select the same columns
2. Rename the datetime columns to a common name
3. Add a `taxi_type` column so we can tell them apart after the union

In [ ]:
trips_clean = (
    trips
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(F.col("pickup_datetime").isNotNull())
)

trips_clean.show(5)

---
## 2. Basic Transformations

### 2a — Select, rename, filter

In [ ]:
trips_enriched = (
    trips_clean
    .withColumn(
        "duration_min",
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60
    )
    .withColumn(
        "speed_mph",
        F.round(F.col("trip_distance") / (F.col("duration_min") / 60), 2)
    )
    .withColumn(
        "tip_pct",
        F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 1)
    )
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn("pickup_dow",  F.dayofweek("pickup_datetime"))  # 1=Sun, 7=Sat
    .withColumn(
        "payment_label",
        F.when(F.col("payment_type") == 1, "Credit card")
         .when(F.col("payment_type") == 2, "Cash")
         .when(F.col("payment_type") == 3, "No charge")
         .otherwise("Other")
    )
    .filter(F.col("duration_min").between(1, 120))
    .filter(F.col("speed_mph").between(1, 80))
)

trips_enriched.select("taxi_type", "duration_min", "speed_mph", "tip_pct", "pickup_hour", "payment_label").show(10)

### 2b — Derived columns with `withColumn`

In [ ]:
trips_enriched = (
    trips_clean
    # Duration in minutes
    .withColumn(
        "duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    # Speed in mph
    .withColumn(
        "speed_mph",
        F.round(F.col("trip_distance") / (F.col("duration_min") / 60), 2)
    )
    # Tip percentage
    .withColumn(
        "tip_pct",
        F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 1)
    )
    # Pickup hour + day of week
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow",  F.dayofweek("tpep_pickup_datetime"))  # 1=Sun, 7=Sat
    # Readable payment type
    .withColumn(
        "payment_label",
        F.when(F.col("payment_type") == 1, "Credit card")
         .when(F.col("payment_type") == 2, "Cash")
         .when(F.col("payment_type") == 3, "No charge")
         .otherwise("Other")
    )
    .filter(F.col("duration_min").between(1, 120))
    .filter(F.col("speed_mph").between(1, 80))
)

trips_enriched.select("duration_min", "speed_mph", "tip_pct", "pickup_hour", "payment_label").show(10)

### 2c — UDF vs built-in functions

UDFs are convenient but run row-by-row in Python and cannot be optimized by Spark's Catalyst.  
Always prefer built-in `functions` — use UDFs only when there is no built-in equivalent.

In [ ]:
# UDF version (slow — Python call per row)
def classify_trip_udf(distance):
    if distance is None:
        return "unknown"
    if distance < 1:   return "short"
    if distance < 5:   return "medium"
    return "long"

classify_udf = F.udf(classify_trip_udf, StringType())

t0 = time.time()
trips_enriched.withColumn("trip_type", classify_udf("trip_distance")).count()
print(f"UDF:     {time.time()-t0:.2f}s")

# Built-in version (fast — JVM, Catalyst-optimized)
classify_builtin = (
    F.when(F.col("trip_distance") < 1, "short")
     .when(F.col("trip_distance") < 5, "medium")
     .otherwise("long")
)

t0 = time.time()
trips_enriched.withColumn("trip_type", classify_builtin).count()
print(f"Built-in: {time.time()-t0:.2f}s")

w_day = (
    Window
    .partitionBy(F.to_date("pickup_datetime"))
    .orderBy("pickup_datetime")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

(
    trips_enriched
    .withColumn("running_total", F.sum("fare_amount").over(w_day))
    .select("pickup_datetime", "fare_amount", "running_total")
    .show(10)
)

In [ ]:
# Rank trips by fare within each pickup hour
w_hour = Window.partitionBy("pickup_hour").orderBy(F.desc("fare_amount"))

top_fares = (
    trips_enriched
    .withColumn("rank_in_hour",     F.rank().over(w_hour))
    .withColumn("dense_rank",       F.dense_rank().over(w_hour))
    .withColumn("avg_fare_in_hour", F.avg("fare_amount").over(Window.partitionBy("pickup_hour")))
    .filter(F.col("rank_in_hour") <= 3)
    .select("pickup_hour", "fare_amount", "avg_fare_in_hour", "rank_in_hour", "dense_rank")
    .orderBy("pickup_hour", "rank_in_hour")
)

top_fares.show(12)

In [ ]:
# Running total of fare per day (ordered by pickup time)
w_day = (
    Window
    .partitionBy(F.to_date("tpep_pickup_datetime"))
    .orderBy("tpep_pickup_datetime")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

(
    trips_enriched
    .withColumn("running_total", F.sum("fare_amount").over(w_day))
    .select("tpep_pickup_datetime", "fare_amount", "running_total")
    .show(10)
)

---
## 4. Join Strategies

Spark has three main physical join strategies. Understanding which one fires — and why — is key to writing fast joins.

### 4a — Broadcast Hash Join (BHJ)

**When**: one side is small (default threshold: 10 MB, tunable via `spark.sql.autoBroadcastJoinThreshold`).  
**How**: the small table is copied to every executor — zero shuffle on the large table.  
**Best for**: fact × small dimension (e.g., trips × zones).

In [ ]:
# zones is ~300 rows — Spark will broadcast it automatically
# but we can force it with a hint to make the intent explicit:

trips_with_zones = (
    trips_enriched
    .join(
        F.broadcast(zones.withColumnRenamed("LocationID", "PULocationID")
                        .withColumnRenamed("Zone", "pickup_zone")
                        .withColumnRenamed("Borough", "pickup_borough")
                        .select("PULocationID", "pickup_zone", "pickup_borough")),
        on="PULocationID",
        how="left"
    )
)

trips_with_zones.select("PULocationID", "pickup_zone", "pickup_borough", "fare_amount").show(5)

print("\n=== Plan — look for BroadcastHashJoin ===")
trips_with_zones.explain(mode="formatted")

### 4b — Sort-Merge Join (SMJ)

**When**: both sides are large (above broadcast threshold).  
**How**: both sides are shuffled by join key, then sorted, then merged.  
**Cost**: two shuffles (one per side) — the most expensive but most general strategy.  
**Best for**: large × large joins.

In [ ]:
# Disable broadcast to force SMJ so we can inspect the plan
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Join trips with itself on a key — simulates two large tables
high_tip  = trips_enriched.filter(F.col("tip_pct") > 20).select("PULocationID", "tip_pct")
all_trips = trips_enriched.select("PULocationID", "fare_amount")

smj_result = high_tip.join(all_trips, on="PULocationID", how="inner")

print("=== Plan — look for SortMergeJoin ===")
smj_result.explain(mode="formatted")

# Restore default
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

### 4c — Shuffle Hash Join (SHJ)

**When**: one side is small enough to build a hash table in memory but too large to broadcast.  
**How**: smaller side is shuffled + hashed; larger side probes the hash table.  
**Advantage over SMJ**: no sort step.  
**Risk**: OOM if the build side is larger than executor memory.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")

shj_result = high_tip.join(all_trips, on="PULocationID", how="inner")

print("=== Plan — look for ShuffledHashJoin ===")
shj_result.explain(mode="formatted")

# Restore
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")

### 4d — Join strategy summary

| Strategy | Both sides shuffled? | Sort required? | Use when |
|---|---|---|---|
| **Broadcast Hash Join** | No (small side broadcast) | No | Small × Large (< 10 MB) |
| **Shuffle Hash Join** | Yes (both) | No | Medium × Large (fits in RAM) |
| **Sort-Merge Join** | Yes (both) | Yes | Large × Large (default safe choice) |

### 4e — Join type semantics (inner / left / anti)

Same physical strategy, different row-filtering logic.

In [ ]:
# Which pickup zones had NO high-tip trips? — anti join
zones_simple = zones.select(F.col("LocationID").alias("PULocationID"), "Zone")
high_tip_zones = trips_enriched.filter(F.col("tip_pct") > 20).select("PULocationID").distinct()

no_high_tip_zones = zones_simple.join(high_tip_zones, on="PULocationID", how="left_anti")
no_high_tip_zones.show(10)
print("Zones with no high-tip trips:", no_high_tip_zones.count())

---
## 5. Partitioning

Partitioning controls how data is distributed across executors.  
Wrong partitioning = stragglers, OOM, or tiny tasks.

### 5a — repartition vs coalesce

In [ ]:
print("Default partitions:", trips_enriched.rdd.getNumPartitions())

# repartition — full shuffle, balances data evenly, can increase OR decrease partition count
repartitioned = trips_enriched.repartition(20)
print("After repartition(20):", repartitioned.rdd.getNumPartitions())

# coalesce — avoids shuffle, only decreases count, may create uneven partitions
coalesced = trips_enriched.coalesce(4)
print("After coalesce(4):    ", coalesced.rdd.getNumPartitions())

In [ ]:
# Partition by a key column — all rows with the same key go to the same partition
# Useful before a join or groupBy on that key
by_location = trips_enriched.repartition(20, "PULocationID")

# Inspect partition sizes (expensive — do this only for diagnosis)
sizes = by_location.withColumn("part", F.spark_partition_id()).groupBy("part").count().orderBy("part")
sizes.show(10)
print("Min rows in a partition:", sizes.agg(F.min("count")).collect()[0][0])
print("Max rows in a partition:", sizes.agg(F.max("count")).collect()[0][0])

### 5b — Skew: what happens when one partition is much larger

In [ ]:
# Payment type is very skewed — most rides are credit card
trips_enriched.groupBy("payment_type").count().orderBy(F.desc("count")).show()

# Repartitioning by a skewed column produces uneven partitions
by_payment = trips_enriched.repartition(10, "payment_type")
(
    by_payment
    .withColumn("part", F.spark_partition_id())
    .groupBy("part").count()
    .orderBy(F.desc("count"))
    .show()
)

### 5c — Write partitioning (partition pruning at read time)

In [ ]:
output_partitioned = f"{MY_PREFIX}/output/trips_by_borough"

# Add borough column, then partition the written Parquet by it
trips_with_borough = (
    trips_enriched
    .join(
        F.broadcast(
            zones.select(
                F.col("LocationID").alias("PULocationID"),
                F.col("Borough").alias("pickup_borough")
            )
        ),
        on="PULocationID",
        how="left"
    )
    .filter(F.col("pickup_borough").isNotNull())
)

trips_with_borough.write \
    .partitionBy("pickup_borough") \
    .mode("overwrite") \
    .parquet(output_partitioned)

print(f"Written to {output_partitioned}")

In [ ]:
# Read back — filter on the partition column triggers partition pruning
manhattan_only = spark.read.parquet(output_partitioned).filter(F.col("pickup_borough") == "Manhattan")

print("=== Plan — look for PartitionFilters ===")
manhattan_only.explain(mode="formatted")

# Spark reads ONLY the Manhattan partition directory — not the full dataset
print("Manhattan trips:", manhattan_only.count())

---
## 6. Caching

`cache()` / `persist()` stores a DataFrame in memory (or disk) so it's not recomputed on every action.  
Cache only when a DataFrame is **reused multiple times** and **expensive to compute**.

In [ ]:
# trips_enriched is used repeatedly below — good cache candidate
trips_enriched.cache()

# First action triggers compute + cache population
t0 = time.time()
n = trips_enriched.count()
print(f"First action (cold): {time.time()-t0:.2f}s  rows={n:,}")

# Second action reads from cache
t0 = time.time()
n = trips_enriched.count()
print(f"Second action (hot): {time.time()-t0:.2f}s  rows={n:,}")

In [ ]:
# Multiple aggregations on the same cached DataFrame — only read once
avg_fare    = trips_enriched.agg(F.avg("fare_amount")).collect()[0][0]
avg_tip     = trips_enriched.agg(F.avg("tip_pct")).collect()[0][0]
avg_speed   = trips_enriched.agg(F.avg("speed_mph")).collect()[0][0]

print(f"Avg fare:  ${avg_fare:.2f}")
print(f"Avg tip:    {avg_tip:.1f}%")
print(f"Avg speed: {avg_speed:.1f} mph")

# Release when done — free up executor memory
trips_enriched.unpersist()

---
## 7. Putting it together — Borough-level trip report

In [ ]:
borough_stats = (
    trips_with_borough
    .groupBy("pickup_borough")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("fare_amount"),  2).alias("avg_fare"),
        F.round(F.avg("tip_pct"),      1).alias("avg_tip_pct"),
        F.round(F.avg("trip_distance"),2).alias("avg_distance_mi"),
        F.round(F.avg("speed_mph"),    1).alias("avg_speed_mph"),
        F.round(F.avg("duration_min"), 1).alias("avg_duration_min"),
    )
    .orderBy(F.desc("total_trips"))
)

borough_stats.show()

In [ ]:
# Hourly demand heatmap (borough × hour)
heatmap = (
    trips_with_borough
    .groupBy("pickup_borough", "pickup_hour")
    .count()
    .orderBy("pickup_borough", "pickup_hour")
)

heatmap.show(24)

---
## Homework

---

### Q1 — Top pickup zones
Find the **top 10 pickup zones** (Zone name, not ID) by number of trips.  
Use a join with the zones table and select the Zone column.

In [ ]:
# Q1


### Q2 — Rush hour tip behavior
Compare the **average tip percentage** during rush hours (7–9 AM and 5–7 PM) vs. off-peak hours.  
Show: `time_category` (rush / off-peak), `avg_tip_pct`, `trip_count`.

In [ ]:
# Q2


### Q3 — Window rank per borough
For each borough, find the **top 3 pickup zones** by average fare.  
Use a window function (not a self-join or subquery).  
Show: `pickup_borough`, `pickup_zone`, `avg_fare`, `rank`.

In [ ]:
# Q3


### Q4 — Join strategy experiment
1. Disable auto-broadcast (`spark.sql.autoBroadcastJoinThreshold = -1`)
2. Join trips with zones without a broadcast hint
3. Run `.explain()` — which join strategy does Spark pick?
4. Re-enable broadcast and run `.explain()` again
5. In the Markdown cell below, explain what changed in the plan and why.

In [ ]:
# Q4


**Q4 written answer**:  
_your explanation here_

### Q5 — Partition write + pruning
1. Write `trips_enriched` partitioned by **`pickup_hour`** to `s3://<bucket>/<your-name>/homework/trips_by_hour/`
2. Read it back filtering only `pickup_hour == 8` (rush hour)
3. Call `.explain()` and confirm `PartitionFilters` appears in the plan
4. How many files are in the `pickup_hour=8` directory vs the total dataset?

In [ ]:
# Q5


### Q6 — Bonus: skew handling
The `PULocationID` column is skewed (some zones have far more trips than others).  
1. Confirm the skew: show the top 10 locations by trip count
2. Join `trips_enriched` with zones on `PULocationID` without a broadcast hint and with `autoBroadcastJoinThreshold = -1`
3. Add a **salting key** to the skewed side: append a random integer 0–4 to `PULocationID` to create `salted_key`. Explode the zones side with the same 0–4 suffix. Join on `salted_key`.
4. Compare the partition size distribution before and after salting using `spark_partition_id()`.

In [ ]:
# Q6 bonus


In [ ]:
spark.stop()